# Baseline — Pet Image Rotation Prediction

**Competition:** each circle-cropped pet image was rotated by 0°, 90°, 180° or 270°
(labels 1–4). Predict the rotation.

- **Task:** 4-class classification (1400 train / test images)
- **Metric:** `score = 1 / (1 + circular_mse)` — rotation is circular, so predicting 4
  when the truth is 1 is only one step away
- **Kaggle link:** _TODO: add link_

**Approach:** frozen ImageNet ResNet-18 features + Logistic Regression. Pretrained
features encode orientation cues (sky position, face orientation) surprisingly well.

In [1]:
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)

(1400, 3) (850, 2)


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
weights = ResNet18_Weights.IMAGENET1K_V1
backbone = resnet18(weights=weights)
backbone.fc = torch.nn.Identity()
backbone.eval().to(device)
preprocess = weights.transforms()

@torch.no_grad()
def extract_features(paths, batch_size=32):
    feats = []
    for i in range(0, len(paths), batch_size):
        batch = [preprocess(Image.open(f"{DATA_DIR}/{p}").convert("RGB"))
                 for p in paths[i:i+batch_size]]
        feats.append(backbone(torch.stack(batch).to(device)).cpu().numpy())
    return np.vstack(feats)

X  = extract_features(train["image_path"].tolist())
Xt = extract_features(test["image_path"].tolist())
print(X.shape, Xt.shape)

(1400, 512) (850, 512)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

y = train["rotation_label"].values
clf = LogisticRegression(max_iter=3000)

def circular_mse(y_true, y_pred):
    d = np.abs(y_true - y_pred) % 4
    d = np.minimum(d, 4 - d)          # circular distance on 4 labels
    return np.mean(d.astype(float) ** 2)

oof = cross_val_predict(clf, X, y, cv=5)
cmse = circular_mse(y, oof)
acc = (oof == y).mean()
print(f"CV accuracy: {acc:.4f}   circular MSE: {cmse:.4f}   score: {1/(1+cmse):.4f}")

CV accuracy: 0.6429   circular MSE: 0.9486   score: 0.5132


In [4]:
clf.fit(X, y)
sub = pd.DataFrame({"id": test["id"], "rotation_label": clf.predict(Xt)})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,rotation_label
0,test_00000,4
1,test_00001,3
2,test_00002,3
3,test_00003,2
4,test_00004,1


## Ideas to improve

- **Self-supervised trick:** you can create unlimited extra training data — take any
  image, rotate it yourself by a known amount, and you have a new labeled sample.
- Fine-tune the full CNN with this augmentation on GPU.
- Use test-time augmentation: predict on all 4 rotations of a test image and pick the
  consistent answer.
